# 02 — NLP Severity Classification Model
---
**What this notebook does:**
Trains an NLP model that reads the free-text `INSPECTION_TEXT_DESCRIPTION`
from an OHS inspection record and predicts one of four severity levels:
`Critical` / `High` / `Medium` / `Low`

**Why NLP?**
An inspector's written narrative contains rich semantic context that structured
fields cannot capture — words like *"fatal"*, *"Stop Work Order"*, *"near-miss"*
carry very different risk signals. A sentence-level transformer model captures
this meaning far better than keyword matching or bag-of-words approaches.

**Architecture:**
```
Inspection Text
     │
     ▼
 SentenceTransformer (all-MiniLM-L6-v2)
 — 384-dimensional semantic embedding —
     │
     ▼
 LogisticRegression classifier
     │
     ▼
 severity_level + confidence + probabilities
```

**Why this model combination?**
- `all-MiniLM-L6-v2` is a compact (80MB) but highly capable sentence transformer
  that maps text to dense semantic vectors in 384 dimensions
- `LogisticRegression` on top of normalized embeddings is fast, interpretable,
  and works very well when the embedding space is already semantically rich
- The combination trains in seconds on 400 records and serves predictions in <100ms

**Output:** Registered MLflow model → `ohs_severity_classifier` (Version 1)

**Runtime:** Serverless &nbsp;|&nbsp; **Prerequisite:** Notebook 01 must have run successfully

## Cell 1 — Install Dependencies
`sentence-transformers` is not pre-installed on Databricks Serverless.
This installs the HuggingFace sentence-transformers library which pulls in
PyTorch, transformers, and tokenizers as dependencies.
**First run takes 3–5 minutes.** Subsequent runs use the cached packages.

In [0]:
%pip install sentence-transformers --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import sys
print(f"Current Python version: {sys.version}")

Current Python version: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]


## Cell 2 — Imports
Import the ML, NLP, and MLflow libraries needed for this notebook.
Note that `mlflow.pyfunc` is used to create a **custom model class** — this
lets us bundle the transformer + classifier together as a single serveable unit.

In [0]:
import pandas as pd
import numpy as np
import pickle                           # for serialising the classifier artifacts

# Scikit-learn components
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# HuggingFace sentence transformer — generates semantic embeddings from text
from sentence_transformers import SentenceTransformer

# MLflow — experiment tracking, model packaging, and registry
import mlflow
import mlflow.pyfunc

/local_disk0/.ephemeral_nfs/envs/pythonEnv-8886a6d9-e8d7-4c6e-9e35-70d69d98829c/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


## Cell 3 — Load Training Data
Load the synthetic dataset created in Notebook 01 from the Delta table.
We convert it to a pandas DataFrame because scikit-learn and
sentence-transformers work with numpy arrays, not Spark DataFrames.

In [0]:
# Load from Unity Catalog Delta table — requires Notebook 01 to have run first
df = spark.table("workspace.ohs_data.synthetic_inspections").toPandas()

print(f"Loaded {len(df)} records from workspace.ohs_data.synthetic_inspections")
print(f"\nColumns available: {list(df.columns)}")
print(f"\nSeverity distribution (our target labels):")
print(df["SEVERITY_LEVEL"].value_counts())
print(f"\nSample inspection text:")
print(df["INSPECTION_TEXT_DESCRIPTION"].iloc[0])

Loaded 100 records from workspace.ohs_data.synthetic_inspections

Columns available: ['FIELD_VISIT_DATE', 'FIELD_VISIT_TYPE', 'CASE_TYPE', 'CASE_STATUS', 'WORKPLACE_ID', 'WORKPLACE_NAME_AT_FV_TIME', 'WORKPLACE_ADDRESS_AT_FV_TIME', 'POSTAL_CODE', 'PRIMARY_NAICS', 'NAICS_DESCRIPTION', 'CONTRAVENER_ROLE', 'CONTRAVENER_ORG_ID', 'CONTRAVENER_NAME', 'ORDER_TYPE', 'ORDER_STATUS', 'CASE_ACT', 'ACT_REG_ID', 'ACT_REGULATION_NAME', 'SEC', 'SUBSEC', 'CLAUSE', 'INSPECTION_TEXT_DESCRIPTION', 'SEVERITY_LEVEL', 'RISK_SCORE']

Severity distribution (our target labels):
SEVERITY_LEVEL
Medium      35
High        29
Low         27
Critical     9
Name: count, dtype: int64

Sample inspection text:
Multiple fall hazards identified on multi-storey construction site. Floor openings unguarded with no covers or safety nets. Ladders improperly secured and visibly damaged. Two workers observed at height without personal fall arrest systems engaged. Site safety coordinator absent at time of inspection.


## Cell 4 — Prepare Labels and Train/Test Split

**Label encoding:**
`LabelEncoder` converts string class names to integers for scikit-learn.
Example: `Critical→0, High→1, Low→2, Medium→3` (alphabetical order by default)
We keep the encoder object so we can reverse-map predictions back to strings.

**Stratified split:**
`stratify=y_enc` ensures the same class proportions appear in both
train and test sets — important when one class (Critical) is rare.

In [0]:
# Extract the text column (model input) and severity column (target label)
X_text = df["INSPECTION_TEXT_DESCRIPTION"].tolist()   # list of strings
y_raw  = df["SEVERITY_LEVEL"].values                  # numpy array of class names

# Encode string labels to integers (scikit-learn requirement)
le     = LabelEncoder()
y_enc  = le.fit_transform(y_raw)   # e.g. 'Critical'→0, 'High'→1, 'Low'→2, 'Medium'→3
classes = le.classes_.tolist()      # ['Critical', 'High', 'Low', 'Medium']

print(f"Classes:       {classes}")
print(f"Label mapping: { {c: i for i, c in enumerate(classes)} }")

# Stratified 80/20 train-test split — stratify preserves class proportions
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y_enc,
    test_size=0.2,
    random_state=42,
    stratify=y_enc,    # ensures each split has the same % Critical/High/Medium/Low
)

print(f"\nTrain size: {len(X_train_text)} records")
print(f"Test size:  {len(X_test_text)} records")

Classes:       ['Critical', 'High', 'Low', 'Medium']
Label mapping: {'Critical': 0, 'High': 1, 'Low': 2, 'Medium': 3}

Train size: 80 records
Test size:  20 records


## Cell 5 — Generate Sentence Embeddings

This is the core NLP step — converting variable-length text into fixed-size
numeric vectors that capture **semantic meaning**.

**How `all-MiniLM-L6-v2` works:**
The model is a distilled version of BERT with 6 transformer layers.
It maps each input text to a 384-dimensional vector where semantically
similar sentences are close together in vector space.
For example, *"fatal fall"* and *"worker died from height"* will have
similar embeddings even though they share no keywords.

**`normalize_embeddings=True`:**
Projects all vectors onto a unit sphere. This makes dot products
equivalent to cosine similarity, which LogisticRegression handles very well.

**First run:** downloads the model (~90MB) from HuggingFace.
**Subsequent runs:** loads from the Databricks package cache instantly.

**Save to `/tmp/`:**
We save the model weights to `/tmp/ohs_sbert_model` so MLflow can package
them as an artifact. Note: `/dbfs/` is NOT available on Serverless — use `/tmp/`.

In [0]:
MODEL_NAME = "all-MiniLM-L6-v2"   # 384-dim, ~80MB, excellent speed/quality tradeoff

# /dbfs/ is NOT available on Serverless Compute — use /tmp/ instead.
# MLflow copies the directory into the run artifact store at log_model time,
# so the local /tmp path only needs to exist during this notebook run.
SBERT_SAVE_PATH = "/tmp/ohs_sbert_model"

print(f"Loading SentenceTransformer: {MODEL_NAME}")
print("(Downloads ~90MB from HuggingFace on first run — cached after that)")
encoder = SentenceTransformer(MODEL_NAME)

# Encode all training texts into 384-dimensional semantic vectors
# batch_size=32 processes 32 texts at a time to balance speed and memory
print("\nEncoding training texts into embeddings...")
X_train_emb = encoder.encode(
    X_train_text,
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True,   # unit-normalise for cosine-similarity behaviour
)

# Encode test texts using the same encoder (no fitting — transforms only)
print("\nEncoding test texts into embeddings...")
X_test_emb = encoder.encode(
    X_test_text,
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True,
)

print(f"\nEmbedding shape: {X_train_emb.shape}")   # expected: (400, 384)
print(f"Each text → {X_train_emb.shape[1]}-dimensional vector")

# Save the transformer weights so MLflow can bundle them with the registered model
encoder.save(SBERT_SAVE_PATH)
print(f"\nTransformer weights saved to: {SBERT_SAVE_PATH}")

Loading SentenceTransformer: all-MiniLM-L6-v2
(Downloads ~90MB from HuggingFace on first run — cached after that)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...


Encoding test texts into embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding shape: (80, 384)
Each text → 384-dimensional vector


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Transformer weights saved to: /tmp/ohs_sbert_model


## Cell 6 — Train the Severity Classifier

Trains a `LogisticRegression` classifier on top of the 384-dim embeddings.

**Why Logistic Regression?**
- The embeddings already encode rich semantic features — we just need a
  linear boundary in that high-dimensional space
- Fast to train, interpretable, low risk of overfitting on 400 samples
- Works well with normalised embeddings (unit vectors)

**5-Fold Cross-Validation:**
Trains and tests on 5 different splits to get a reliable accuracy estimate
that isn't sensitive to a single lucky/unlucky split.

In [0]:
# ── 5-fold cross-validation for reliable accuracy estimate ───────────────────
# This trains 5 versions of the model on different train/val splits
# and reports the mean ± std accuracy — a more robust measure than single split
clf_cv  = LogisticRegression(max_iter=1000, random_state=42, C=1.0, solver="lbfgs")
cv_scores = cross_val_score(clf_cv, X_train_emb, y_train, cv=5, scoring="accuracy")
print(f"5-Fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Individual fold scores: {[round(s, 4) for s in cv_scores]}")

# ── Final model trained on the full training set ──────────────────────────────
# C=1.0 is the regularisation strength — higher = less regularisation
# solver='lbfgs' is efficient for small multi-class problems
clf = LogisticRegression(max_iter=1000, random_state=42, C=1.0, solver="lbfgs")
clf.fit(X_train_emb, y_train)

# ── Evaluate on the held-out test set ────────────────────────────────────────
y_pred = clf.predict(X_test_emb)
acc    = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=classes, output_dict=True)

print(f"\nTest Set Accuracy: {acc:.4f}")
print("\nFull Classification Report (precision, recall, F1 per class):")
print(classification_report(y_test, y_pred, target_names=classes))

print("Confusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred))
print(f"Classes order: {classes}")

5-Fold CV Accuracy: 0.9000 ± 0.0500
Individual fold scores: [np.float64(0.9375), np.float64(0.9375), np.float64(0.9375), np.float64(0.8125), np.float64(0.875)]

Test Set Accuracy: 0.9500

Full Classification Report (precision, recall, F1 per class):
              precision    recall  f1-score   support

    Critical       1.00      0.50      0.67         2
        High       0.86      1.00      0.92         6
         Low       1.00      1.00      1.00         5
      Medium       1.00      1.00      1.00         7

    accuracy                           0.95        20
   macro avg       0.96      0.88      0.90        20
weighted avg       0.96      0.95      0.94        20

Confusion Matrix (rows=actual, cols=predicted):
[[1 1 0 0]
 [0 6 0 0]
 [0 0 5 0]
 [0 0 0 7]]
Classes order: ['Critical', 'High', 'Low', 'Medium']


## Cell 7 — Define the Custom MLflow PythonModel

We wrap the transformer + classifier into a single `mlflow.pyfunc.PythonModel`.
This is the key step that makes the model **serveable** — when Databricks
Model Serving loads this model, it calls `load_context()` once to initialise,
then calls `predict()` for every incoming API request.

**Why a custom pyfunc and not `mlflow.sklearn.log_model`?**
Because our model has two components: the HuggingFace transformer (not sklearn)
and the sklearn classifier. The custom class bundles both together so the
serving endpoint receives raw text and returns structured JSON — no preprocessing
needed on the client side.

**Serving endpoint I/O format:**
```
Input:  {"dataframe_records": [{"inspection_text": "Worker fell from scaffold..."}]}
Output: [{"severity_level": "Critical", "confidence": 0.92, "probabilities": {...}}]
```

In [0]:
class SeverityClassifierModel(mlflow.pyfunc.PythonModel):
    """
    Custom MLflow pyfunc model that bundles:
      - SentenceTransformer (all-MiniLM-L6-v2) for text → embedding
      - LogisticRegression classifier for embedding → severity level

    Loaded once at serving startup via load_context().
    Called for every prediction request via predict().
    """

    def load_context(self, context: mlflow.pyfunc.PythonModelContext) -> None:
        """
        Called once when the model is loaded by the serving endpoint.
        Loads both the transformer weights and the pickled classifier.
        `context.artifacts` maps artifact keys to their local file paths.
        """
        from sentence_transformers import SentenceTransformer
        import pickle

        # Load transformer from the saved weights directory
        # context.artifacts["sentence_transformer"] = local path to /tmp/ohs_sbert_model
        self.encoder = SentenceTransformer(
            context.artifacts["sentence_transformer"],
            device="cpu",   # CPU is fine; switch to "cuda" on a GPU serving endpoint
        )
        self.encoder.max_seq_length = 256   # cap very long texts to avoid OOM

        # Load the pickled sklearn classifier + label encoder
        with open(context.artifacts["classifier_artifacts"], "rb") as fh:
            arts = pickle.load(fh)

        self.classifier    = arts["classifier"]     # trained LogisticRegression
        self.label_encoder = arts["label_encoder"]  # maps int indices back to class names
        self.classes       = arts["classes"]        # ['Critical', 'High', 'Low', 'Medium']

    def predict(
        self,
        context: mlflow.pyfunc.PythonModelContext,
        model_input: pd.DataFrame,
    ):
        """
        Called for every prediction request.
        Accepts a DataFrame with column 'inspection_text'.
        Returns a list of dicts with severity_level, confidence, and probabilities.
        """
        # Accept both DataFrame (batch) and dict (single record) inputs
        if isinstance(model_input, pd.DataFrame):
            texts = model_input["inspection_text"].tolist()
        elif isinstance(model_input, dict):
            texts = [model_input["inspection_text"]]
        else:
            texts = list(model_input)

        # Step 1: Convert texts to semantic embeddings
        embeddings = self.encoder.encode(
            texts,
            show_progress_bar=False,
            batch_size=32,
            normalize_embeddings=True,
        )

        # Step 2: Predict class index and probability distribution
        pred_enc  = self.classifier.predict(embeddings)          # integer class indices
        proba_mat = self.classifier.predict_proba(embeddings)    # shape (n_texts, 4)

        # Step 3: Build the structured output for each prediction
        results = []
        for p_enc, probs in zip(pred_enc, proba_mat):
            severity  = self.label_encoder.inverse_transform([p_enc])[0]  # int → class name
            prob_dict = {
                cls: round(float(p), 4)
                for cls, p in zip(self.classes, probs)
            }
            results.append({
                "severity_level": severity,           # e.g. "Critical"
                "confidence":     round(float(max(probs)), 4),  # highest probability
                "probabilities":  prob_dict,          # all 4 class probabilities
            })

        return results

## Cell 8 — Log to MLflow and Register the Model

This cell does three things:
1. **Logs** the run (params, metrics, artifacts) to the MLflow experiment `/OHS/severity_nlp_model`
2. **Packages** the transformer directory + classifier pickle into a single pyfunc model
3. **Registers** the model as `ohs_severity_classifier` in the MLflow Model Registry

After this cell, go to **Machine Learning → Models** in the sidebar to see the registered model.

In [0]:
# Set the MLflow experiment — creates it if it doesn't exist
# Using a path under the user's workspace directory to avoid permission issues
mlflow.set_experiment("/Workspace/Users/nishit.rathod@ontario.ca/SDS_Assignment_243232/OHS_severity_nlp_model")

with mlflow.start_run(run_name="severity_sbert_lr_v1") as run:

    # ── Log hyperparameters ───────────────────────────────────────────────────
    # These are recorded so you can compare different runs in the Experiments UI
    mlflow.log_params({
        "encoder_model":  MODEL_NAME,               # which transformer was used
        "embedding_dim":  X_train_emb.shape[1],     # 384
        "classifier":     "LogisticRegression",
        "LR_C":           1.0,                       # regularisation strength
        "LR_solver":      "lbfgs",
        "normalize_emb":  True,
        "train_samples":  len(X_train_text),
        "test_samples":   len(X_test_text),
    })

    # ── Log evaluation metrics ────────────────────────────────────────────────
    mlflow.log_metric("accuracy",           acc)
    mlflow.log_metric("cv_accuracy_mean",   cv_scores.mean())
    mlflow.log_metric("cv_accuracy_std",    cv_scores.std())

    # Log per-class F1, precision, recall for detailed tracking
    for cls_name in classes:
        if cls_name in report:
            mlflow.log_metric(f"f1_{cls_name}",        report[cls_name]["f1-score"])
            mlflow.log_metric(f"precision_{cls_name}", report[cls_name]["precision"])
            mlflow.log_metric(f"recall_{cls_name}",    report[cls_name]["recall"])

    # ── Serialise classifier + label encoder into a single pickle file ────────
    # This file is one of the two artifacts bundled into the MLflow model
    clf_arts_path = "/tmp/severity_clf_artifacts.pkl"
    with open(clf_arts_path, "wb") as fh:
        pickle.dump({
            "classifier":    clf,    # trained LogisticRegression
            "label_encoder": le,     # LabelEncoder (string ↔ int mapping)
            "classes":       classes # ['Critical', 'High', 'Low', 'Medium']
        }, fh)

    # ── Define the conda environment for the serving container ────────────────
    # Databricks uses this to recreate the Python environment when serving
    conda_env = {
        "channels": ["defaults", "conda-forge"],
        "dependencies": [
            "python=3.12", "pip",
            {"pip": [
                "sentence-transformers>=2.2.0",
                "scikit-learn>=1.2.0",
                "torch>=2.0.0",
                "pandas>=1.5.0",
                "numpy>=1.23.0",
                "mlflow>=2.0.0",
            ]},
        ],
        "name": "ohs_severity_env",
    }

    # ── Create model signature for Unity Catalog registration ─────────────────
    # Unity Catalog requires both input and output types to be specified
    # Generate sample prediction using the trained components to infer schema
    sample_input = pd.DataFrame({"inspection_text": [X_test_text[0]]})
    sample_emb = encoder.encode([X_test_text[0]], normalize_embeddings=True)
    sample_pred_enc = clf.predict(sample_emb)
    sample_proba = clf.predict_proba(sample_emb)[0]
    sample_output = [{
        "severity_level": le.inverse_transform(sample_pred_enc)[0],
        "confidence": float(max(sample_proba)),
        "probabilities": {cls: float(p) for cls, p in zip(classes, sample_proba)}
    }]
    signature = mlflow.models.infer_signature(sample_input, sample_output)

    # ── Log the pyfunc model with BOTH artifacts bundled ─────────────────────
    # artifacts dict maps names → local paths
    # MLflow copies these files into the run's artifact store
    mlflow.pyfunc.log_model(
        artifact_path="severity_model",

        python_model=SeverityClassifierModel(),     # our custom class above

        artifacts={
            "sentence_transformer":  SBERT_SAVE_PATH,  # the /tmp/ohs_sbert_model dir
            "classifier_artifacts":  clf_arts_path,     # the .pkl file
        },

        conda_env=conda_env,

        signature=signature,              # Required for Unity Catalog registration
        input_example=sample_input,       # Shows expected input format in UI

        # Registering here creates Version 1 in the MLflow Model Registry
        registered_model_name="ohs_severity_classifier",
    )

    print(f"\n✓  MLflow run ID : {run.info.run_id}")
    print(f"✓  Registered as : ohs_severity_classifier (Version 1)")
    print(f"   → Go to Machine Learning → Models to see it")

2026/05/31 19:39:13 INFO mlflow.tracking.fluent: Experiment with name '/Workspace/Users/nishit.rathod@ontario.ca/SDS_Assignment_243232/OHS_severity_nlp_model' does not exist. Creating a new experiment.
2026/05/31 19:39:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-71dc0835-779f.cloud.databricks.com/ml/experiments/3583728162249089/models/m-248f84ba28e5473d83bf2cde3ab7d6fc?o=3652072476397248
2026/05/31 19:39:16 INFO mlflow.pyfunc: Validating input example against model signature


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Successfully registered model 'workspace.default.ohs_severity_classifier'.


Uploading artifacts:   0%|          | 0/21 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.ohs_severity_classifier': https://dbc-71dc0835-779f.cloud.databricks.com/explore/data/models/workspace/default/ohs_severity_classifier/version/1?o=3652072476397248



✓  MLflow run ID : 21eda9b098bc410c9e61dd3749335172
✓  Registered as : ohs_severity_classifier (Version 1)
   → Go to Machine Learning → Models to see it


## Cell 9 — Validate the Registered Model

Loads the model back from the registry and runs 5 test predictions.
This confirms the model was serialised correctly and can be loaded
the same way the serving endpoint will load it.

**Expected output:** Severity predictions that match the text content.
- Text about fatal falls → `Critical`
- Text about overdue fire extinguisher tags → `Low`

In [0]:
# Load the model back from the registry — same code path the serving endpoint uses
# Unity Catalog requires explicit version number or alias (not "latest")
loaded_model = mlflow.pyfunc.load_model("models:/workspace.default.ohs_severity_classifier/1")

# Run validation predictions on 5 hand-crafted test cases
validation_texts = pd.DataFrame({"inspection_text": [
    "Fatal fall from scaffold. Worker found on ground. No PPE present. Stop Work Order issued.",
    "Fire extinguisher tags overdue by 60 days. Extinguishers in good condition. Employer cooperative.",
    "Workers operating forklifts without certification. Multiple near-misses in last 30 days.",
    "Critical injury from unguarded rotating machinery. No lockout tagout procedures in place.",
    "Missing WHMIS safety data sheets for 3 chemicals on site. Training records incomplete.",
]})

predictions = loaded_model.predict(validation_texts)

print("=== VALIDATION PREDICTIONS ===\n")
for text, pred in zip(validation_texts["inspection_text"].tolist(), predictions):
    # Use emoji icons for quick visual check of predictions
    icon = {"Critical": "🔴", "High": "🟠", "Medium": "🟡", "Low": "🟢"}.get(
        pred["severity_level"], "⚪"
    )
    print(f"{icon}  [{pred['severity_level']}]  (confidence: {pred['confidence']:.1%})")
    print(f"    Text: {text[:80]}...")
    print()

print("✓  Model validation complete — ready to create the serving endpoint")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

=== VALIDATION PREDICTIONS ===

🔴  [Critical]  (confidence: 37.5%)
    Text: Fatal fall from scaffold. Worker found on ground. No PPE present. Stop Work Orde...

🟢  [Low]  (confidence: 51.3%)
    Text: Fire extinguisher tags overdue by 60 days. Extinguishers in good condition. Empl...

🟠  [High]  (confidence: 39.0%)
    Text: Workers operating forklifts without certification. Multiple near-misses in last ...

🟠  [High]  (confidence: 30.2%)
    Text: Critical injury from unguarded rotating machinery. No lockout tagout procedures ...

🟡  [Medium]  (confidence: 56.1%)
    Text: Missing WHMIS safety data sheets for 3 chemicals on site. Training records incom...

✓  Model validation complete — ready to create the serving endpoint
